# Latihan Python untuk Data Engineering

Notebook ini untuk belajar konsep di materi persiapan Data Engineer Supervisor, langsung dengan kode yang bisa dijalankan.

**Cara pakai**

- Buka di Google Colab: *File > Upload notebook*, pilih file ini.
- Jalankan sel satu per satu dari atas dengan **Shift + Enter**. Urutannya penting karena sel bawah memakai hasil sel atas.
- Setelah sebuah sel jalan, baca outputnya dan coba ubah sedikit kodenya. Itu cara belajar paling cepat.

**Isi**

0. Cara membaca Python
1. pandas dengan data dummy penjualan obat
2. Incremental load (pipeline dari database sumber ke warehouse)
3. PySpark: filter, groupBy, window function, join, menulis Parquet
4. Latihan mandiri
5. Kunci jawaban (jangan diintip dulu)

## 0. Cara membaca Python

Lima pola yang paling sering muncul:

| Pola | Arti |
| --- | --- |
| `x = 10` | simpan nilai ke variabel `x` |
| `nama(a, b)` | panggil fungsi `nama` dengan input `a` dan `b` |
| `objek.method()` | panggil fungsi milik objek, misalnya `teks.upper()` |
| indentasi (4 spasi) | menandai isi sebuah blok (`for`, `if`, `def`) |
| `# komentar` | catatan, tidak dijalankan |

In [ ]:
# Variabel dan tipe data
nama = "Farrel"          # string (teks)
umur = 23                # integer (bilangan bulat)
ipk = 3.30               # float (bilangan desimal)
lulus = True             # boolean

print(nama, umur, ipk, lulus)
print(type(ipk))

In [ ]:
# List: kumpulan nilai berurutan
produk = ["Bodrex", "Hemaviton", "Oskadon"]
print(produk[0])        # indeks mulai dari 0
print(len(produk))      # jumlah item

# Dictionary: pasangan kunci -> nilai (mirip satu baris tabel)
order = {"order_id": 1, "produk": "Bodrex", "qty": 10}
print(order["qty"])

In [ ]:
# Fungsi: kumpulan langkah yang diberi nama dan bisa dipakai ulang
def hitung_total(qty, harga):
    total = qty * harga
    return total            # nilai yang dikembalikan ke pemanggil

print(hitung_total(10, 5000))

# Loop dan kondisi
for p in produk:
    if p.startswith("B"):
        print(p, "diawali huruf B")
    else:
        print(p)

## 1. pandas dengan data dummy

pandas adalah library untuk mengolah tabel di Python. Satu tabel disebut **DataFrame**.

Kita buat data dummy penjualan obat dari beberapa distributor. Data ini sengaja diberi sedikit "kotoran" (qty negatif, harga kosong, baris dobel) supaya bisa dipakai latihan data quality.

In [ ]:
import pandas as pd
import random

random.seed(42)   # supaya data dummy selalu sama setiap dijalankan

produk_list = {"Bodrex": 8000, "Hemaviton": 12000, "Oskadon": 6000, "Contrexyn": 7000, "NeoRheumacyl": 15000}
distributor_list = ["PT Sinar", "PT Maju", "PT Sentosa", "PT Abadi"]

rows = []
for i in range(1, 201):
    nama_produk = random.choice(list(produk_list))
    rows.append({
        "order_id": i,
        "tanggal": f"2026-{random.randint(7, 9):02d}-{random.randint(1, 28):02d}",
        "distributor": random.choice(distributor_list),
        "produk": nama_produk,
        "qty": random.randint(5, 100),
        "harga": produk_list[nama_produk],
        "status": random.choice(["completed"] * 8 + ["cancelled"] * 2),
    })

penjualan = pd.DataFrame(rows)

# Sisipkan data kotor
penjualan.loc[5, "qty"] = -3          # qty tidak masuk akal
penjualan.loc[17, "harga"] = None     # harga kosong
penjualan = pd.concat([penjualan, penjualan.iloc[[30]]], ignore_index=True)   # baris dobel

penjualan.head()

In [ ]:
# Mengenal data
print(penjualan.shape)          # (jumlah baris, jumlah kolom)
print(penjualan.dtypes)         # tipe tiap kolom
penjualan.describe()            # statistik ringkas kolom angka

### Operasi dasar dan padanannya di SQL

| pandas | SQL |
| --- | --- |
| `df[df["status"] == "completed"]` | `WHERE status = 'completed'` |
| `df[["produk", "qty"]]` | `SELECT produk, qty` |
| `df.groupby("produk")["qty"].sum()` | `SELECT produk, SUM(qty) ... GROUP BY produk` |
| `df.sort_values("qty", ascending=False)` | `ORDER BY qty DESC` |
| `df.merge(df2, on="distributor")` | `JOIN df2 USING (distributor)` |

In [ ]:
# Kolom baru: revenue = qty x harga
penjualan["revenue"] = penjualan["qty"] * penjualan["harga"]

# WHERE status = 'completed'
selesai = penjualan[penjualan["status"] == "completed"]

# GROUP BY distributor, SUM(revenue), ORDER BY revenue DESC
per_distributor = (selesai.groupby("distributor")["revenue"]
                          .sum()
                          .sort_values(ascending=False))
per_distributor

In [ ]:
# Data quality check sederhana
print("Qty tidak valid (<= 0):", (penjualan["qty"] <= 0).sum())
print("Harga kosong          :", penjualan["harga"].isna().sum())
print("order_id dobel        :", penjualan["order_id"].duplicated().sum())

In [ ]:
# Membersihkan data
bersih = (penjualan
          .drop_duplicates(subset="order_id")     # buang baris dobel
          .dropna(subset=["harga"])               # buang harga kosong
          .query("qty > 0"))                      # buang qty tidak valid

print(len(penjualan), "->", len(bersih), "baris")

## 2. Incremental load: dari database sumber ke warehouse

Skenario: ada database aplikasi (`sumber.db`) yang terus berubah. Kita ingin menyalinnya ke warehouse (`gudang.db`) secara berkala, **hanya mengambil yang berubah**, dan **aman dijalankan ulang** tanpa data dobel.

Kita pakai SQLite karena sudah ada di Python, tidak perlu install database.

Konsep kunci:

- **Watermark**: catatan "sampai kapan data sudah berhasil dimuat".
- **Upsert**: kalau ID sudah ada, update; kalau belum, insert.
- **Idempotent**: dijalankan berkali-kali hasilnya tetap sama.

In [ ]:
import sqlite3
import os

# Mulai dari nol setiap kali sel ini dijalankan
for f in ["sumber.db", "gudang.db"]:
    if os.path.exists(f):
        os.remove(f)

src = sqlite3.connect("sumber.db")    # database aplikasi (sumber)
dwh = sqlite3.connect("gudang.db")    # data warehouse (tujuan)

# Isi database sumber dengan 3 order
src.execute("CREATE TABLE orders (order_id INTEGER, customer TEXT, amount REAL, updated_at TEXT)")
src.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", [
    (1, "Apotek Sehat", 100000, "2026-10-01 10:00:00"),
    (2, "Apotek Jaya",  -5000,  "2026-10-02 09:00:00"),   # amount negatif (data kotor)
    (3, "Apotek Prima", 250000, "2026-10-03 14:30:00"),
])
src.commit()

# Siapkan tabel di warehouse
dwh.execute("CREATE TABLE watermark (tabel TEXT PRIMARY KEY, terakhir TEXT)")
dwh.execute("CREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer TEXT, amount REAL, updated_at TEXT)")
dwh.commit()

pd.read_sql("SELECT * FROM orders", src)

### Fungsi-fungsi pipeline

Keempat fungsi ini **kita tulis sendiri**, bukan bawaan Python. Baca komentarnya baris per baris.

In [ ]:
def get_watermark(tabel):
    """Ambil waktu terakhir load berhasil. Kalau belum pernah, mulai dari 1970."""
    row = dwh.execute("SELECT terakhir FROM watermark WHERE tabel = ?", (tabel,)).fetchone()
    return row[0] if row else "1970-01-01 00:00:00"


def set_watermark(tabel, waktu):
    """Simpan waktu terakhir. ON CONFLICT = kalau sudah ada, update saja."""
    dwh.execute(
        "INSERT INTO watermark VALUES (?, ?) "
        "ON CONFLICT(tabel) DO UPDATE SET terakhir = excluded.terakhir",
        (tabel, waktu))
    dwh.commit()


def transform(df):
    """Bersihkan data sebelum masuk warehouse."""
    df = df.dropna(subset=["order_id"])            # buang baris tanpa ID
    df["amount"] = df["amount"].clip(lower=0)      # amount negatif jadi 0
    return df


def upsert(df):
    """Insert baris baru, update baris yang ID-nya sudah ada."""
    rows = df[["order_id", "customer", "amount", "updated_at"]].itertuples(index=False, name=None)
    dwh.executemany(
        "INSERT INTO orders VALUES (?, ?, ?, ?) "
        "ON CONFLICT(order_id) DO UPDATE SET customer = excluded.customer, "
        "amount = excluded.amount, updated_at = excluded.updated_at",
        rows)
    dwh.commit()

In [ ]:
def jalankan_pipeline():
    last = get_watermark("orders")                 # 1. sampai mana terakhir?
    terbaru = last
    query = "SELECT * FROM orders WHERE updated_at > ?"

    # 2. ambil hanya yang berubah, per potongan (chunk) supaya hemat memori
    for chunk in pd.read_sql(query, src, params=[last], chunksize=50_000):
        if chunk.empty:
            continue
        clean = transform(chunk)                   # 3. bersihkan
        upsert(clean)                              # 4. masukkan ke warehouse
        terbaru = max(terbaru, clean["updated_at"].max())

    # 5. simpan watermark PALING AKHIR, setelah semua sukses.
    #    Pakai updated_at terbesar, bukan jam sekarang, supaya tidak ada data yang terlewat.
    set_watermark("orders", terbaru)
    print("Watermark sekarang:", terbaru)

### Percobaan 1: load pertama

In [ ]:
jalankan_pipeline()
pd.read_sql("SELECT * FROM orders", dwh)

### Percobaan 2: jalankan ulang tanpa perubahan

Hasilnya harus tetap 3 baris, tidak ada yang dobel. Inilah arti **idempotent**.

In [ ]:
jalankan_pipeline()
pd.read_sql("SELECT * FROM orders", dwh)

### Percobaan 3: ada perubahan di sumber

Order 1 diubah dan ada order baru. Pipeline hanya mengambil 2 baris itu.

In [ ]:
src.execute("UPDATE orders SET amount = 999000, updated_at = '2026-10-05 08:00:00' WHERE order_id = 1")
src.execute("INSERT INTO orders VALUES (4, 'Apotek Baru', 75000, '2026-10-05 09:15:00')")
src.commit()

jalankan_pipeline()
pd.read_sql("SELECT * FROM orders", dwh)

## 3. PySpark

Spark dipakai kalau data terlalu besar untuk satu komputer. Konsepnya mirip pandas, tetapi:

- Data dibagi ke banyak **partisi** dan diproses paralel.
- Spark itu **lazy**: perintah seperti `filter` dan `groupBy` baru dijalankan saat ada **action** seperti `show()`, `count()`, atau `write`.

Di Colab, install dulu (sekitar 1 menit):

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# SparkSession = pintu masuk ke Spark. Di Databricks sudah dibuat otomatis.
spark = SparkSession.builder.appName("latihan").getOrCreate()

# Ubah DataFrame pandas yang sudah bersih menjadi DataFrame Spark
df = spark.createDataFrame(bersih)
df.printSchema()
df.show(5)

### Filter, groupBy, agg

Sama dengan contoh di materi. Padanan SQL-nya:

```sql
SELECT distributor, tanggal AS day, SUM(revenue) AS revenue
FROM penjualan
WHERE status = 'completed'
GROUP BY distributor, tanggal
```

In [ ]:
harian = (df.filter(F.col("status") == "completed")                          # WHERE
            .groupBy("distributor", F.to_date("tanggal").alias("day"))       # GROUP BY
            .agg(F.sum("revenue").alias("revenue")))                         # SUM(...) AS revenue

# Sampai baris ini belum ada yang dihitung (lazy). show() yang memicu eksekusi.
harian.orderBy("day").show(10)

### Menulis ke Parquet dengan partisi

`partitionBy("day")` membuat satu subfolder per tanggal. Query yang memfilter tanggal nanti cukup membaca folder yang perlu saja (**partition pruning**).

In [ ]:
harian.write.mode("overwrite").partitionBy("day").parquet("output/daily_revenue")

# Lihat struktur foldernya
import os
print(sorted(os.listdir("output/daily_revenue"))[:5])

# Baca lagi, hanya untuk satu tanggal
spark.read.parquet("output/daily_revenue").filter(F.col("day") == "2026-08-15").show()

### Window function: transaksi terakhir tiap distributor

Pertanyaan interview yang sering muncul. Padanan SQL:

```sql
SELECT * FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY distributor ORDER BY tanggal DESC) AS rn
  FROM penjualan
) WHERE rn = 1
```

In [ ]:
from pyspark.sql.window import Window

w = Window.partitionBy("distributor").orderBy(F.col("tanggal").desc(), F.col("order_id").desc())

terakhir = (df.withColumn("rn", F.row_number().over(w))
              .filter(F.col("rn") == 1)
              .drop("rn"))
terakhir.show()

### Menulis SQL langsung di Spark

Kalau lebih nyaman dengan SQL, daftarkan DataFrame sebagai tabel sementara.

In [ ]:
df.createOrReplaceTempView("penjualan")

spark.sql("""
    SELECT produk, SUM(qty) AS total_qty, ROUND(AVG(harga)) AS harga_rata
    FROM penjualan
    WHERE status = 'completed'
    GROUP BY produk
    ORDER BY total_qty DESC
""").show()

### Broadcast join

Tabel dimensi kecil (data distributor) dikirim ke semua executor, jadi tabel besar tidak perlu di-shuffle.

In [ ]:
dim_distributor = spark.createDataFrame([
    ("PT Sinar",   "Jawa Barat",  "Bandung"),
    ("PT Maju",    "DKI Jakarta", "Jakarta"),
    ("PT Sentosa", "Jawa Timur",  "Surabaya"),
    ("PT Abadi",   "Jawa Tengah", "Semarang"),
], ["distributor", "region", "kota"])

per_region = (df.join(F.broadcast(dim_distributor), on="distributor", how="left")
                .filter(F.col("status") == "completed")
                .groupBy("region")
                .agg(F.sum("revenue").alias("revenue"), F.countDistinct("order_id").alias("jumlah_order"))
                .orderBy(F.col("revenue").desc()))
per_region.show()

# explain() menampilkan rencana eksekusi. Cari kata BroadcastHashJoin.
per_region.explain()

## 4. Latihan mandiri

Kerjakan di sel kosong di bawah tiap soal. Kalau macet, cek bagian 5.

**Soal 1 (pandas).** Dari `bersih`, hitung total `qty` per produk untuk order `completed`, urutkan dari terbesar.

**Soal 2 (pandas).** Tambahkan kolom `bulan` (misalnya `2026-08`) dari kolom `tanggal`, lalu hitung revenue per bulan.

**Soal 3 (incremental load).** Tambahkan 2 order baru ke `sumber.db`, jalankan `jalankan_pipeline()`, lalu pastikan warehouse berisi 6 baris.

**Soal 4 (PySpark).** Hitung revenue per bulan per produk. Petunjuk: `F.date_format(F.to_date("tanggal"), "yyyy-MM")`.

**Soal 5 (PySpark, data quality).** Pakai DataFrame Spark dari data **kotor** (`penjualan`), hitung berapa baris yang qty-nya <= 0 atau harganya kosong. Petunjuk: operator `|` untuk OR. Hati-hati, harga kosong dari pandas bisa masuk ke Spark sebagai `NaN`, bukan `null`.

In [ ]:
# Soal 1

In [ ]:
# Soal 2

In [ ]:
# Soal 3

In [ ]:
# Soal 4

In [ ]:
# Soal 5

## 5. Kunci jawaban

Coba kerjakan sendiri dulu, baru cocokkan.

In [ ]:
# Jawaban 1
(bersih[bersih["status"] == "completed"]
    .groupby("produk")["qty"].sum()
    .sort_values(ascending=False))

In [ ]:
# Jawaban 2
bersih["bulan"] = bersih["tanggal"].str[:7]       # ambil 7 karakter pertama: 2026-08
bersih.groupby("bulan")["revenue"].sum()

In [ ]:
# Jawaban 3
src.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", [
    (5, "Apotek Mawar", 120000, "2026-10-06 10:00:00"),
    (6, "Apotek Melati", 90000, "2026-10-06 11:30:00"),
])
src.commit()
jalankan_pipeline()
print(pd.read_sql("SELECT COUNT(*) AS jumlah FROM orders", dwh))

In [ ]:
# Jawaban 4
(df.filter(F.col("status") == "completed")
   .withColumn("bulan", F.date_format(F.to_date("tanggal"), "yyyy-MM"))
   .groupBy("bulan", "produk")
   .agg(F.sum("revenue").alias("revenue"))
   .orderBy("bulan", F.col("revenue").desc())
   .show())

In [ ]:
# Jawaban 5
kotor = spark.createDataFrame(penjualan)
# Harga kosong dari pandas masuk ke Spark sebagai NaN (Not a Number), bukan null.
# isNull() saja tidak menangkap NaN, jadi cek keduanya.
tidak_valid = kotor.filter(
    (F.col("qty") <= 0) | F.col("harga").isNull() | F.isnan("harga")
)
print("Baris tidak valid:", tidak_valid.count())
tidak_valid.select("order_id", "qty", "harga").show()

## Langkah berikutnya

- Ulangi bagian 2 sampai bisa menulis keempat fungsinya tanpa melihat.
- Coba Apache Iceberg dengan Spark lewat quickstart resminya: https://iceberg.apache.org/spark-quickstart/
- Tulis ulang proyek Hadoop MapReduce-mu dalam PySpark memakai pola di bagian 3.